# Tokenization

In [1]:
import tiktoken

# Load the encoding used by GPT-4 and recent models
encoding = tiktoken.get_encoding("cl100k_base")

# Define the input sentences
english_sentence = "Artificial intelligence is changing the world."
vietnamese_sentence = "Trí tuệ nhân tạo đang thay đổi thế giới."

# Encode the sentences to get the list of token IDs
english_token_ids = encoding.encode(english_sentence)
vietnamese_token_ids = encoding.encode(vietnamese_sentence)

# Decode each token ID back to text to see exactly how the model splits the strings
english_token_strings = [encoding.decode([token]) for token in english_token_ids]
vietnamese_token_strings = [encoding.decode([token]) for token in vietnamese_token_ids]

# Print the results for the English sentence
print("--- English Tokenization ---")
print(f"Original Text: {english_sentence}")
print(f"Token IDs: {english_token_ids}")
print(f"Text Chunks: {english_token_strings}")
print(f"Total Tokens: {len(english_token_ids)}")
print("\n")

# Print the results for the Vietnamese sentence
print("--- Vietnamese Tokenization ---")
print(f"Original Text: {vietnamese_sentence}")
print(f"Token IDs: {vietnamese_token_ids}")
print(f"Text Chunks: {vietnamese_token_strings}")
print(f"Total Tokens: {len(vietnamese_token_ids)}")

--- English Tokenization ---
Original Text: Artificial intelligence is changing the world.
Token IDs: [9470, 16895, 11478, 374, 10223, 279, 1917, 13]
Text Chunks: ['Art', 'ificial', ' intelligence', ' is', ' changing', ' the', ' world', '.']
Total Tokens: 8


--- Vietnamese Tokenization ---
Original Text: Trí tuệ nhân tạo đang thay đổi thế giới.
Token IDs: [1305, 2483, 9964, 26298, 20921, 40492, 259, 89416, 15199, 526, 270, 352, 15199, 98616, 270, 27160, 13845, 53047, 13]
Text Chunks: ['Tr', 'í', ' tu', 'ệ', ' nh', 'ân', ' t', 'ạo', ' đ', 'ang', ' th', 'ay', ' đ', 'ổi', ' th', 'ế', ' gi', 'ới', '.']
Total Tokens: 19


1. Context window là giải pháp giúp LLM ghi nhớ được nội dung của cuộc hội thoại trước đó
2. Hallucination: là việc mà câu trả lời của mô hình LLM không đúng sự thật làm mà vẫn trả lời và diễn đạt hợp lí

# Text Embedding Model

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load a pre-trained multilingual embedding model suitable for Vietnamese
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Define three test sentences
sentence_a = "Tôi rất thích nuôi chó cưng trong nhà."
sentence_b = "Mèo là một loài động vật vô cùng đáng yêu."
sentence_c = "Hôm nay thời tiết bên ngoài mưa rất to."

sentences = [sentence_a, sentence_b, sentence_c]

# Generate embeddings for the sentences
print("Generating embeddings...")
embeddings = model.encode(sentences)

# Extract individual vectors
vector_a = embeddings[0]
vector_b = embeddings[1]
vector_c = embeddings[2]

# Print the shape and a small sample of the vector to show it is just an array of numbers
print("\n--- Vector Dimensions ---")
print(f"Each sentence is converted into a vector with {len(vector_a)} dimensions.")
print(f"Sample of Vector A (first 5 numbers): {vector_a[:5]}")

# Calculate Cosine Similarity
# Reshape vectors to 2D arrays as required by scikit-learn
vector_a_2d = vector_a.reshape(1, -1)
vector_b_2d = vector_b.reshape(1, -1)
vector_c_2d = vector_c.reshape(1, -1)

# Compute similarity scores
similarity_a_b = cosine_similarity(vector_a_2d, vector_b_2d)[0][0]
similarity_a_c = cosine_similarity(vector_a_2d, vector_c_2d)[0][0]

# Print the final results
print("\n--- Semantic Similarity Results ---")
print(f"Sentence A: '{sentence_a}'")
print(f"Sentence B: '{sentence_b}'")
print(f"Sentence C: '{sentence_c}'")
print("-" * 30)
print(f"Similarity between A and B (Animals): {similarity_a_b:.4f}")
print(f"Similarity between A and C (Animal vs Weather): {similarity_a_c:.4f}")

c:\Users\tabao\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Generating embeddings...

--- Vector Dimensions ---
Each sentence is converted into a vector with 384 dimensions.
Sample of Vector A (first 5 numbers): [ 0.14158382 -0.38071302 -0.10997102  0.26502103 -0.26710415]

--- Semantic Similarity Results ---
Sentence A: 'Tôi rất thích nuôi chó cưng trong nhà.'
Sentence B: 'Mèo là một loài động vật vô cùng đáng yêu.'
Sentence C: 'Hôm nay thời tiết bên ngoài mưa rất to.'
------------------------------
Similarity between A and B (Animals): 0.4551
Similarity between A and C (Animal vs Weather): -0.0000


In [3]:
from transformers import AutoTokenizer, AutoModel
import torch

# 1. Load the pre-trained multilingual BERT model and its tokenizer
model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Define the input sentence
text = "Con đường thành công không hề dễ dàng."

# 2. The Tokenization process specific to BERT
print("--- Step 1: Tokenization ---")
inputs = tokenizer(text, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(f"Original text: '{text}'")
print(f"Tokens generated by BERT: {tokens}")

# 3. Pass the tokenized inputs through the BERT deep neural network
print("\n--- Step 2: Passing through the BERT model ---")
with torch.no_grad():
    outputs = model(**inputs)

# outputs.last_hidden_state contains the contextualized vectors for ALL tokens
word_vectors = outputs.last_hidden_state
print(f"Shape of the output matrix: {word_vectors.shape}")
# Explanation: The shape is typically [1, Number_of_tokens, 768].
# This means: 1 sentence, N tokens, and each token is a 768-dimensional vector.

# 4. Extract the Sentence Embedding (The representative vector for the entire sentence)
print("\n--- Step 3: Extracting the sentence vector ---")
# In BERT's architecture, the first token [CLS] aggregates the semantic meaning of the whole sentence
sentence_embedding = word_vectors[0][0] 

print(f"Dimension of the sentence vector: {sentence_embedding.shape[0]}")
print(f"First 5 values of the sentence vector:\n{sentence_embedding[:5]}")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


--- Step 1: Tokenization ---
Original text: 'Con đường thành công không hề dễ dàng.'
Tokens generated by BERT: ['[CLS]', 'Con', 'đường', 'thành', 'công', 'không', 'h', '##ề', 'dễ', 'dà', '##ng', '.', '[SEP]']

--- Step 2: Passing through the BERT model ---
Shape of the output matrix: torch.Size([1, 13, 768])

--- Step 3: Extracting the sentence vector ---
Dimension of the sentence vector: 768
First 5 values of the sentence vector:
tensor([-0.1260, -0.0427, -0.1276,  0.0918, -0.1847])


# Demo for Basic RAG Knowledge

In [4]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
load_dotenv(override=True)
# 0. Setup environment variable for OpenAI API Key
# Replace 'YOUR_OPENAI_API_KEY' with your actual key
OPENAI__API_KEY = os.environ["OPENAI_API_KEY"] 

# ---------------------------------------------------------
# STEP 1: DATA LOADING
# ---------------------------------------------------------
print("--- Step 1: Data Loading ---")

# Create a mock text file simulating a company policy document
file_path = "mock_policy.txt"
with open(file_path, "w", encoding="utf-8") as file:
    file.write("Nhân viên chính thức được nghỉ phép 12 ngày mỗi năm. "
               "Công ty hỗ trợ tiền ăn trưa 50.000 VNĐ mỗi ngày làm việc. "
               "Bảo hiểm y tế sẽ được đóng đầy đủ sau 2 tháng thử việc.")

# Load the document using LangChain's TextLoader
loader = TextLoader(file_path, encoding="utf-8")
documents = loader.load()
print(f"Successfully loaded {len(documents)} document(s).\n")

# ---------------------------------------------------------
# STEP 2: SPLITTING
# ---------------------------------------------------------
print("--- Step 2: Text Splitting ---")

# Initialize the text splitter to break the document into smaller chunks
# We use a small chunk_size here because our mock document is very short
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=70,
    chunk_overlap=15
)
chunks = text_splitter.split_documents(documents)

print(f"Split the document into {len(chunks)} chunks.")
print(f"Sample of Chunk 1: '{chunks[0].page_content}'")
print(f"Sample of Chunk 2: '{chunks[1].page_content}'\n")

# ---------------------------------------------------------
# STEP 3 & 4: EMBEDDING & STORING
# ---------------------------------------------------------
print("--- Step 3 & 4: Embedding and Storing ---")

# Initialize the embedding model
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

# Create a local Chroma vector database, embed the chunks, and store them
vector_db = Chroma.from_documents(documents=chunks, embedding=embeddings_model)
print("Successfully embedded all chunks and stored them in ChromaDB.\n")

# ---------------------------------------------------------
# STEP 5: RETRIEVING
# ---------------------------------------------------------
print("--- Step 5: Retrieving ---")

# Define the question that the user wants to ask
user_query = "Mức phụ cấp ăn trưa của công ty hiện tại là bao nhiêu tiền?"
print(f"User Query: '{user_query}'")

# Perform a semantic search in the vector database
# k=2 means we want to retrieve the top 2 most relevant chunks
retrieved_docs = vector_db.similarity_search(user_query, k=2)

print("Retrieved chunks matching the query:")
for i, doc in enumerate(retrieved_docs):
    print(f"  - Retrieved Context [{i+1}]: '{doc.page_content}'")
print("\n")

# ---------------------------------------------------------
# STEP 6: GENERATING
# ---------------------------------------------------------
print("--- Step 6: Generating ---")

# Initialize the Large Language Model for text generation
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Define a strict prompt template to prevent hallucination
template = """
You are a helpful HR assistant. Answer the user's question based ONLY on the provided context.
If the context does not contain the answer, simply output: "Tôi không tìm thấy thông tin này trong tài liệu."

Context provided from the database:
{context}

User Question: {question}

Answer strictly in Vietnamese:
"""
prompt = PromptTemplate.from_template(template)

# Combine the retrieved chunks into a single text block
context_text = "\n".join([doc.page_content for doc in retrieved_docs])

# Format the final prompt string by injecting the context and the user query
final_prompt = prompt.format(context=context_text, question=user_query)

# Send the prompt to the LLM and get the response
response = llm.invoke(final_prompt)

print("Final Generated Answer:")
print(response.content)

# Cleanup: Remove the mock text file after execution
if os.path.exists(file_path):
    os.remove(file_path)

--- Step 1: Data Loading ---
Successfully loaded 1 document(s).

--- Step 2: Text Splitting ---
Split the document into 3 chunks.
Sample of Chunk 1: 'Nhân viên chính thức được nghỉ phép 12 ngày mỗi năm. Công ty hỗ trợ'
Sample of Chunk 2: 'Công ty hỗ trợ tiền ăn trưa 50.000 VNĐ mỗi ngày làm việc. Bảo hiểm y'

--- Step 3 & 4: Embedding and Storing ---
Successfully embedded all chunks and stored them in ChromaDB.

--- Step 5: Retrieving ---
User Query: 'Mức phụ cấp ăn trưa của công ty hiện tại là bao nhiêu tiền?'
Retrieved chunks matching the query:
  - Retrieved Context [1]: 'Công ty hỗ trợ tiền ăn trưa 50.000 VNĐ mỗi ngày làm việc. Bảo hiểm y'
  - Retrieved Context [2]: 'Nhân viên chính thức được nghỉ phép 12 ngày mỗi năm. Công ty hỗ trợ'


--- Step 6: Generating ---
Final Generated Answer:
Mức phụ cấp ăn trưa của công ty hiện tại là 50.000 VNĐ mỗi ngày làm việc.
